# tsundoku-research-agent を Colab で動かす

Inoreader の「後で読む」を週次でまとめ・採点・要約してレポートにする。保存先は Google ドライブの `MyDrive/tsundoku`。

必要なもの：OpenAI の API キー（Colab のシークレット `OPENAI_API_KEY`）、Inoreader のアプリ ID／キー（`INOREADER_APP_ID`、`INOREADER_APP_KEY`。Pro プラン）。

**最初の1回**は API キーなしで `demo` を動かして流れを確かめられます（セル3）。

In [ ]:
# 1. インストール（GitHub から）
%pip install -q "git+https://github.com/nagayoshi3/tsundoku-research-agent.git"


In [ ]:
# 2. 保存先：Google ドライブ。チェックポイント（SQLite）はローカル /content に置き、止まるたびにドライブへ複製する
from google.colab import drive
drive.mount("/content/drive")

WORKDIR = "/content/drive/MyDrive/tsundoku"
LOCAL_DIR = "/content/tsundoku"


In [ ]:
# 3. API キーなしのデモ（サンプル記事＋偽 LLM）。流れと出力の形を確かめる
from tsundoku import cli
from tsundoku.config import setup

setup(fake_llm=True, sample_data=True, workdir=WORKDIR, local_dir=LOCAL_DIR)
cli.AUTO_APPROVE = True            # 承認①②を自動で通す
cli.run_interactive("weekly", "2026-W40", fresh=True)


In [ ]:
# 4. 本番の設定。自分用の config.yaml をドライブに置いておくと、それを読む（なければ同梱の既定値）
import os
from tsundoku import cli
from tsundoku.config import setup

cfg = os.path.join(WORKDIR, "config.yaml")
setup(fake_llm=False, sample_data=False, workdir=WORKDIR, local_dir=LOCAL_DIR,
      config_path=cfg if os.path.exists(cfg) else None)   # キーは Colab のシークレットから読む
cli.AUTO_APPROVE = False


In [ ]:
# 5. 初回だけ：Inoreader の認証（ブラウザで許可 → URL を貼る）と、保存日時の意味を確かめる
# from tsundoku.sources.inoreader import check_inoreader
# check_inoreader()


In [ ]:
# 6. 実行。期間は日付・週・月・last・-1 で指定。fresh=True は前回の結果を捨てて最初から
cli.run_interactive("weekly", "2026-10-15")
# cli.run_interactive("weekly", "2026-10-15", fresh=True)
# cli.run_interactive("monthly", "2026-09")


レポートは `MyDrive/tsundoku/report-weekly-2026-W42.md`（Markdown）と `.docx`（Word）。経路の記録は `weekly-2026-W42-trace.csv`。